In [10]:
#https://pygithub.readthedocs.io/en/stable/introduction.html
from github import Github
# Authentication is defined via github.Auth
from github import Auth
import pandas as pd
import numpy as np
import json 
from datetime import datetime, date
from collections import Counter
import plotly.express as px
import time
import pickle
intrinsic_people = ["@aaronchongth","@akash-roboticist","@andreasBihlmaier","@arjo129","@audrow","@azeey","@damon-oss","@faximan","@jennuine","@koonpeng","@kscottz","@luca-della-vedova","@marcoag","@mbordignon-intrinsic","@methylDragon","@mjcarroll","@mjeronimo","@mxgrey","@nuclearsandwich-ai","@quarkytale","@scpeters","@sloretz","@tfoote","@udaya2899","@xiyuoh","@Yadunund"]
org_name = "gazebosim"
this_year = 2024
last_year = 2023

In [2]:
# Grab the access token
with open('./tokens.json',"r") as json_data:
    tokens = json.loads(json_data.read())
    json_data.close()

auth = Auth.Token(tokens["Github"])

# Public Web Github
gh = Github(auth=auth)

In [3]:
org = gh.get_organization(org_name)
repos = org.get_repos()

In [4]:
def extract_contributions(gh, repo_name, start_date, end_date):
# Get github repo level stats for two date ranges
    repo = gh.get_repo(repo_name)
    prs = repo.get_pulls(state='closed', sort='created')
    year = []
    results = {}
    count = 0
    for pr in prs:
        if start_date < pr.closed_at.date() < end_date:
            year.append(pr)
            count += 1
            
    results["repo"] = repo_name
    results["prs"] = year
    results["start_date"] = start_date    
    results["end_date"] = end_date    
    results["users"] = []
    results["handles"] = []
    results["add"] = 0
    results["del"] = 0 
    results["comments"] = 0
    results["files"] = 0
    
    for pr in year:
        results["users"].append(pr.user.name)
        results["handles"].append(pr.user.login)
        results["files"] += pr.changed_files 
        results["del"] += pr.deletions
        results["add"] += pr.additions
        results["comments"] += pr.review_comments

    results["total_prs"] = count
    results["total_users"] = len(set(results["users"]))

    return results

In [5]:
# Create a list of github repos for an org
full_repo_list = []
i = 0
has_repos = True
while has_repos:
    next_repos = repos.get_page(i)
    if len(next_repos) > 0:
        i += 1
        full_repo_list += next_repos
    else:
        has_repos = False
        
print(full_repo_list)
print(len(full_repo_list))

[Repository(full_name="gazebosim/ros_gz"), Repository(full_name="gazebosim/design"), Repository(full_name="gazebosim/gz-rviz"), Repository(full_name="gazebosim/sdf_tutorials"), Repository(full_name="gazebosim/sdformat"), Repository(full_name="gazebosim/docs"), Repository(full_name="gazebosim/ign-acropolis"), Repository(full_name="gazebosim/ign-blueprint"), Repository(full_name="gazebosim/gz-citadel"), Repository(full_name="gazebosim/gz-cmake"), Repository(full_name="gazebosim/gz-common"), Repository(full_name="gazebosim/gz-fuel-tools"), Repository(full_name="gazebosim/gz-sim"), Repository(full_name="gazebosim/ign-go1"), Repository(full_name="gazebosim/gz-gui"), Repository(full_name="gazebosim/gz-launch"), Repository(full_name="gazebosim/gz-math"), Repository(full_name="gazebosim/gz-msgs"), Repository(full_name="gazebosim/gz-physics"), Repository(full_name="gazebosim/gz-plugin"), Repository(full_name="gazebosim/gz-rendering"), Repository(full_name="gazebosim/gz-rndf"), Repository(full_n

In [6]:
this_year_start = date(this_year, 1, 1)
this_year_end = date(this_year, 12, 31)
last_year_start = date(last_year, 1, 1)
last_year_end = date(last_year, 12, 31)
full_org_results = {}
fname = 'github_stats_{0}_{1}-{2}.pkl'.format(org_name,this_year,last_year)
for repo in full_repo_list:
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,this_year_start,this_year_end))
    this_year_results = extract_contributions(gh, repo.full_name, this_year_start, this_year_end)
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,last_year_start,last_year_end))
    last_year_results = extract_contributions(gh, repo.full_name, last_year_start, last_year_end)
    full_org_results[repo.name] = {}
    full_org_results[repo.name][this_year] = this_year_results
    full_org_results[repo.name][last_year] = last_year_results
    with open(fname,"wb") as file:
        pickle.dump(full_org_results, file)
        print("Wrote: {0}".format(fname))
    print("-----------------------------")



Extracting data for gazebosim/ros_gz from 2024-01-01 to 2024-12-31
Extracting data for gazebosim/ros_gz from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="gazebosim")_2024-2023.pkl
-----------------------------
Extracting data for gazebosim/design from 2024-01-01 to 2024-12-31
Extracting data for gazebosim/design from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="gazebosim")_2024-2023.pkl
-----------------------------
Extracting data for gazebosim/gz-rviz from 2024-01-01 to 2024-12-31
Extracting data for gazebosim/gz-rviz from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="gazebosim")_2024-2023.pkl
-----------------------------
Extracting data for gazebosim/sdf_tutorials from 2024-01-01 to 2024-12-31
Extracting data for gazebosim/sdf_tutorials from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="gazebosim")_2024-2023.pkl
-----------------------------
Extracting data for gazebosim/sdformat from 2024-01-01 to 2024-1

Request GET /repositories/255865265/pulls?state=closed&sort=created&page=62 failed with 403: Forbidden
Setting next backoff to 2097.011298s


Extracting data for gazebosim/gz-sim from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="gazebosim")_2024-2023.pkl
-----------------------------
Extracting data for gazebosim/ign-go1 from 2024-01-01 to 2024-12-31
Extracting data for gazebosim/ign-go1 from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="gazebosim")_2024-2023.pkl
-----------------------------
Extracting data for gazebosim/gz-gui from 2024-01-01 to 2024-12-31
Extracting data for gazebosim/gz-gui from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="gazebosim")_2024-2023.pkl
-----------------------------
Extracting data for gazebosim/gz-launch from 2024-01-01 to 2024-12-31
Extracting data for gazebosim/gz-launch from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="gazebosim")_2024-2023.pkl
-----------------------------
Extracting data for gazebosim/gz-math from 2024-01-01 to 2024-12-31
Extracting data for gazebosim/gz-math from 2023-01-01 to 2023-12-31
Wro

Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/domain_bridge from 2024-01-01 to 2024-12-31
Extracting data for ros2/domain_bridge from 2023-01-01 to 2023-12-31
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/ros_network_viz from 2024-01-01 to 2024-12-31
Extracting data for ros2/ros_network_viz from 2023-01-01 to 2023-12-31
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/orocos_kdl_vendor from 2024-01-01 to 2024-12-31
Extracting data for ros2/orocos_kdl_vendor from 2023-01-01 to 2023-12-31
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/netperf from 2024-01-01 to 2024-12-31
Extracting data for ros2/netperf from 2023-01-01 to 2023-12-31
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/rcl_content_filter_fallback from 2024-01-01 to 2024-12-31
Extracting data for ros2/rcl_content_filter_fallback from 2023-01-01 t

In [8]:
out =  None
with open(fname, 'rb') as file:
        out = pickle.load(file)      
print(len(out.keys()))
print(out.keys())

54
dict_keys(['ros_gz', 'design', 'gz-rviz', 'sdf_tutorials', 'sdformat', 'docs', 'ign-acropolis', 'ign-blueprint', 'gz-citadel', 'gz-cmake', 'gz-common', 'gz-fuel-tools', 'gz-sim', 'ign-go1', 'gz-gui', 'gz-launch', 'gz-math', 'gz-msgs', 'gz-physics', 'gz-plugin', 'gz-rendering', 'gz-rndf', 'gz-sensors', 'gz-tools', 'gz-transport', 'gazebo-classic', 'testing', 'gz-bazel', 'ign-dome', 'gz-edifice', 'gz-utils', '.github', 'gz-fortress', 'fortress_demo', 'gz-garden', 'gz-omni', 'gz-omni-meta', 'ci-test', 'gz-mujoco', 'gz-usd', 'gz_pkg_create', 'garden_demo', 'garden-tutorial-party', 'ros_gz_project_template', 'gz-chrono', 'gz-test', 'gz-harmonic', 'harmonic_demo', 'gazebo_test_cases', 'gz-ionic', 'ionic_demo', 'rules_gazebo', 'connect-samples', 'gz-jetty'])


In [11]:
# Do full org aggregation
to_agg = ["users","add","del","files"]

full_results = {}
full_results[this_year] = {}
full_results[last_year] = {}

first = True
for key in full_org_results.keys():
    if first:
        full_results[this_year] = {k: full_org_results[key][this_year][k] for k in to_agg}
        full_results[last_year] = {k: full_org_results[key][last_year][k] for k in to_agg}
        full_results[this_year]["prs"] = len(full_org_results[key][this_year]["prs"])
        full_results[last_year]["prs"] = len(full_org_results[key][last_year]["prs"])
        first = False
    else:        
        for a in to_agg:
            full_results[this_year][a] += full_org_results[key][this_year][a]
            full_results[last_year][a] += full_org_results[key][last_year][a]
            full_results[this_year]["prs"] += len(full_org_results[key][this_year]["prs"])
            full_results[last_year]["prs"] += len(full_org_results[key][last_year]["prs"])

full_results[last_year]["contributors"] = set(full_results[last_year]["users"])
full_results[last_year]["users"] = len(set(full_results[last_year]["users"]))

full_results[this_year]["contributors"] = set(full_results[this_year]["users"])
full_results[this_year]["users"] = len(set(full_results[this_year]["users"]))


print("Results for {0} ==> {1}".format(last_year,this_year))
print("-------------------------")

temp_this = {}
temp_last = {}
temp_change = {}

for k in full_results[this_year].keys():
    if k == "contributors":
        continue
    change = -100*(full_results[last_year][k]-full_results[this_year][k])/full_results[last_year][k]
    temp_this[k] =  full_results[this_year][k]
    temp_last[k] =  full_results[last_year][k]
    temp_change[k] = change
    print("{0:6s}| {1} : {2:<6} | {3} : {4:<6} | {5:4.2f}%".format(k,last_year,full_results[last_year][k],this_year,full_results[this_year][k],change))
print(full_results[this_year]["contributors"])

summary_results = pd.DataFrame(data=[temp_this,temp_last,temp_change])
summary_results.to_csv("{0}-{1}-{2}-GithubContribsSummary.csv".format(org_name,last_year,this_year))

Results for 2023 ==> 2024
-------------------------
users | 2023 : 85     | 2024 : 111    | 30.59%
add   | 2023 : 5825381 | 2024 : 943622 | -83.80%
del   | 2023 : 818013 | 2024 : 670599 | -18.02%
files | 2023 : 16526  | 2024 : 8697   | -47.37%
prs   | 2023 : 6109   | 2024 : 6032   | -1.26%
{'Antoine Van Malleghem', 'yadunund', 'Wassim Kassem', 'Keith Valentin', 'Athena Z.', 'Azmyin Md. Kamal', 'Nate Koenig', 'Yaswanth', 'Carlos Agüero', None, 'Martin Pecka', 'Benjamin Perseghetti', 'Daisuke Sato', 'Gabriel Arjones', 'Serkan Mazlum', 'Jack', 'Ramir Sultanov', 'Mirko Ferrati', 'Michael Beardsworth', 'Sebastian Castro', 'Tatsuro Sakaguchi', 'Vincent', 'Wiktor Bajor ', 'Aryan Jagushte', 'Eloy Briceno', 'Alan', 'Choi Eungyu', 'Udaya Prakash', 'Michael Carroll', 'Silvio Traversaro', 'Nick Oliver', 'Kai Lawrence', 'Sammit Dhar', 'Arjo Chakravarty', 'Nick', 'Alessandro Sofia', 'Nabeel Sherazi', 'Harsh Mahesheka', 'Ryan Govostes', 'Jenn Nguyen', 'Tom Creutz', 'Henry Kotzé', 'David Dorf', 'Heram

In [12]:
target = "add"
agg_result = []
for key in full_org_results.keys():
    a = full_org_results[key][this_year][target]
    b = full_org_results[key][last_year][target]
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("Results for '{0}' across {1} org".format(target,org))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
    
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-NewLines.csv".format(org_name,last_year,this_year))

Results for 'add' across Organization(login="gazebosim") org
-------------------------------------------------------------------
gz-common               | 2023: 24620    | 2024: 420309   | delta: 1607.19%
ionic_demo              | 2023: 0        | 2024: 189121   | delta: 0.00%
gz-sim                  | 2023: 649529   | 2024: 144943   | delta: -77.68%
gz-math                 | 2023: 3963     | 2024: 57041    | delta: 1339.34%
sdformat                | 2023: 47551    | 2024: 31125    | delta: -34.54%
ros_gz                  | 2023: 10076    | 2024: 19231    | delta: 90.86%
gz-physics              | 2023: 31659    | 2024: 16307    | delta: -48.49%
docs                    | 2023: 24552    | 2024: 12458    | delta: -49.26%
gz-rendering            | 2023: 55290    | 2024: 9955     | delta: -81.99%
gz-fuel-tools           | 2023: 6292     | 2024: 8861     | delta: 40.83%
gz-ionic                | 2023: 40       | 2024: 7759     | delta: 19297.50%
gz-transport            | 2023: 53473    | 202

In [14]:
target = "prs"

agg_result = []
for key in full_org_results.keys():
    a = len(full_org_results[key][2024][target])
    b = len(full_org_results[key][2023][target])
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("PR count by year")
print("Results for '{0}' across ROS 2 org".format(target))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-PRS.csv".format(org,last_year,this_year))

PR count by year
Results for 'prs' across ROS 2 org
-------------------------------------------------------------------
ros2_documentation      | 2023: 723      | 2024: 748      | delta: 3.46%
rosbag2                 | 2023: 202      | 2024: 278      | delta: 37.62%
rclcpp                  | 2023: 191      | 2024: 213      | delta: 11.52%
rmw_zenoh               | 2023: 18       | 2024: 177      | delta: 883.33%
rviz                    | 2023: 123      | 2024: 138      | delta: 12.20%
rclpy                   | 2023: 96       | 2024: 128      | delta: 33.33%
geometry2               | 2023: 44       | 2024: 75       | delta: 70.45%
ci                      | 2023: 42       | 2024: 61       | delta: 45.24%
rcl                     | 2023: 82       | 2024: 61       | delta: -25.61%
ros2_tracing            | 2023: 41       | 2024: 54       | delta: 31.71%
ros2cli                 | 2023: 58       | 2024: 53       | delta: -8.62%
rosidl                  | 2023: 39       | 2024: 48       | delta